# Demo — from `demo/config.yaml` to the final document

Input is a YAML file, not settings in this notebook — edit
`demo/config.yaml` (model, extractor, fallback, sample range) and
re-run this notebook top to bottom. Nothing here needs to change.

**How the PDF is read.** A normal PDF goes through `pdfplumber`, which
takes the text layer together with the bold, italic and underline cues
its fonts carry. A PDF whose text layer is missing or broken — a scanned
page, or fonts with no character-to-text mapping — fails a readability
check *before* the model sees it, and is re-read from its page images by
**LightOnOCR** instead (`fallback: auto`). The check is per document, so
clean PDFs never touch the fallback; when one does, `[fallback]` lines
below say which document and why.


## Input — demo/config.yaml, as written on disk


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json

import dmpbridge
print(f'dmpbridge {dmpbridge.__version__}, installed at {Path(dmpbridge.__file__).parent}')

from dmpbridge.core import paths as P
from dmpbridge.evaluation.experiment import ExperimentConfig, Experiment

CONFIG_PATH = Path('demo/config.yaml')
print(CONFIG_PATH.read_text(encoding='utf-8'))


dmpbridge 0.1.0, installed at c:\Users\Nahid\dmpbridge\dmpbridge
# Edit this file, then run:
#   python scripts/run_demo.py
#
# Results (the final labeled DMP document, one JSON per sample) are written
# into demo/output/ when it finishes.

name: Demo run
strategy: wholedoc
provider: ollama
host: http://localhost:11434

model: gemma4:e4b          # any model already pulled in Ollama

# How the PDF is read. pdfplumber handles a normal PDF with a real text layer;
# if that text comes out garbled (a scanned page, or fonts with no character
# mapping) the document is re-read from its page images by LightOnOCR instead.
# Per document — clean PDFs never touch the fallback.
extractor: pdfplumber       # pdfplumber | lightonocr | docling
fallback: auto              # auto = LightOnOCR 

pdf_dir: data/input/pdfs    # expects sample1.pdf, sample2.pdf, ...
sample_start: 1
sample_end: 1              # keep this small for a quick demo run



## Run it


In [2]:
cfg = ExperimentConfig.from_yaml(CONFIG_PATH)
exp = Experiment(cfg)
exp.run()

print(f'\n{cfg.name}: {len(cfg.models)} model(s), {len(cfg.extractors)} extractor(s), '
      f'samples {cfg.sample_start}-{cfg.sample_end}')



Demo run: 1 model(s), 1 extractor(s), samples 1-1


## Output — the final document

Same content `scripts/run_demo.py` copies into `demo/output/final/`; read here
directly from the standard pipeline location so this always reflects the latest run.


In [3]:
model, extractor = cfg.models[0], cfg.extractors[0]
tag = cfg.tag_for(model, extractor)

for n in cfg.sample_range:
    final = P.final_path(tag, n)
    if not final.exists():
        continue
    doc = json.loads(final.read_text(encoding='utf-8'))
    template = doc['narrative']['template']

    print(f'=== sample{n} ===')
    print(f'TITLE: {template["title"]}\n')
    for i, section in enumerate(template['section'], 1):
        print(f'{i}. {section["title"]}')
        for q in section['question']:
            answer = q['answer']['json']['answer']
            print(f'   Q: {q["text"][:70]}')
            print(f'   A: {answer[:90]}{"..." if len(answer) > 90 else ""}')
    print()


=== sample1 ===
TITLE: DATA MANAGEMENT AND SHARING PLAN

1. Element 1: Data Type:
   Q: A. Types and amount of scientific data expected to be generated in the
   A: This secondary data analysis project will analyze deidentified data from 48,218 participan...
   Q: B. Scientific data that will be preserved and shared, and the rational
   A: As this is a secondary data analysis project, we will only be able to publicly share in th...
   Q: C. Metadata, other relevant data, and associated documentation:
   A: In addition to the data described above, code and models will be included in the repositor...
2. Element 2: Related Tools, Software and/or Code:
   Q: Element 2: Related Tools, Software and/or Code:
   A: Data will be analyzed with custom code by our statistical and computer science team. ActiG...
3. Element 3: Standards:
   Q: Element 3: Standards:
   A: The following data will be created as a result of this project: Objective sedentary behavi...
4. Element 4: Data Preservation, Acc

## Save — copy the result into demo/output/

`exp.run()` writes to the pipeline's standard location under `data/output/`.
This copies each stage for the samples in the config into
`demo/output/{labeled,structured,final}/` — the same layout `scripts/run_demo.py` uses.


In [4]:
import shutil

OUTPUT_DIR = Path('demo/output')
STAGES = [('labeled', P.labeled_path), ('structured', P.structured_path), ('final', P.final_path)]

for n in cfg.sample_range:
    for stage, resolve in STAGES:
        src = resolve(tag, n)
        if not src.exists():
            continue
        dest = OUTPUT_DIR / stage / f'sample{n}.json'
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        print(f'{stage:10} -> {dest}')


labeled    -> demo\output\labeled\sample1.json
structured -> demo\output\structured\sample1.json
final      -> demo\output\final\sample1.json


## The final document

The document as saved in `demo/output/final/`, colour-coded by label and
arranged as sections, questions and answers.


In [5]:
from html import escape
from IPython.display import HTML, display

# label -> (colour, text colour); same palette as the pipeline diagram
COLORS = {
    'title':               ('#1E406E', 'white'),
    'section.title':       ('#0F766E', 'white'),
    'section.description': ('#94A3B8', 'black'),
    'question.text':       ('#B45309', 'white'),
    'answer.text':         ('#CBD5E1', 'black'),
}
SIZE   = {'title': '20px', 'section.title': '16px'}
BOLD   = {'title', 'section.title', 'question.text'}


def pill(label, extra=''):
    bg, fg = COLORS.get(label, ('#ddd', 'black'))
    return (f"<span style='background:{bg};color:{fg};font-size:11px;padding:2px 9px;"
            f"border-radius:10px;font-family:monospace'>{escape(label)}{extra}</span>")


def group_html(label, texts):
    """One cell for a run of consecutive blocks with the same label."""
    bg, _ = COLORS.get(label, ('#ddd', 'black'))
    body = ''.join(f"<div style='margin-top:6px;white-space:pre-wrap'>{escape(x)}</div>" for x in texts)
    return (f"<div style='border-left:6px solid {bg};background:{bg}18;padding:6px 12px;"
            f"margin:6px 0;font-family:system-ui,sans-serif;"
            f"font-size:{SIZE.get(label, '13px')};"
            f"font-weight:{'bold' if label in BOLD else 'normal'}'>"
            f"{pill(label, f' x{len(texts)}' if len(texts) > 1 else '')}{body}</div>")


for n in cfg.sample_range:
    # The final document (stage 4) as saved: sections -> questions -> answers.
    final = json.loads(P.final_path(tag, n).read_text(encoding='utf-8'))['narrative']['template']
    parts = [group_html('title', [final['title']])] if final.get('title') else []
    n_questions = 0
    for s in final['section']:
        parts.append(group_html('section.title', [s['title']]))
        if s.get('description'):
            parts.append(group_html('section.description', [s['description']]))
        for q in s.get('question', []):
            n_questions += 1
            parts.append(group_html('question.text', [q.get('text', '')]))
            answer = q.get('answer', {}).get('json', {}).get('answer', '')
            if answer:
                parts.append(group_html('answer.text', [answer]))
    display(HTML(f"<h3 style='font-family:system-ui,sans-serif'>sample{n} - "
                 f"the final document: {len(final['section'])} sections, {n_questions} questions</h3>"
                 + ''.join(parts)))
